This notebook is literally just being used to change the avg_vol module so that it has more flexible path-finding structures. Ignore it.

modularizing avg_vol: need to make it take files from any paths (i.e. paths not hard-coded in...)

Our main thing is getting the weeks loop, so we can make a function (inside the .py file) that looks through the directory for the subject's folder (which either contains weeks as subfolders or week files)

So to deal with eitheir having weeks as files or weeks in subfolders, we can just have the input to avg_vol be the path...?

With the reference file: we can use its parent, but sometimes the files come with week--> subj, so we would need to go up two parents. That doesn't really work.

It should look in base_dir for the file. So need to make a separate function to look for the files.

In [ ]:
from pathlib import Path

base_dir = '/cifs/diedrichsen/data/smarts_cerebellum'

def parent_lookup(file_path, subj_id):
    """
    Looks for subject parent directory of a reference file. Also tracks how many *n* parents there are.

    e.g. If you have a reference file and want to access all other files in its parent (or grandparent, n^th-level parent) directory.

    Inputs:
        file_path (str or Posix path)
        subj_id: str

    Output: subj_id parent folder
    """

    path = Path(file_path)

    for level, parent in enumerate(path.parents, start = 1):
        if parent.name == subj_id:
            return parent, level
    
    print(f'{subj_id} not found in {path}. byeeee')
    return None

In [55]:
subj_id = 'CU_2538'
refT1 = 'W0'
#ref_img = f'{base_dir}/MNISym_GM/{subj_id}/{subj_id}_{refT1}_reslice.nii.gz'
ref_img = f'{base_dir}/anatomicals/{subj_id}/{refT1}/c2{subj_id}_{refT1}_T1.nii'

In [56]:
trial_path, level = parent_lookup(file_path = ref_img, subj_id = subj_id)

In [57]:
trial_path, level

(PosixPath('/cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538'), 2)

In [58]:
def find_week_files(file_path, subj_id, file_naming, subdir = None):
    """
    Finds files in a directory

    If subdir is not None, then looks for structure subj_id/subdir, and searches for week files within subdir

    Inputs:
        file_path (str): reference file
        subj_id (str)
        file_naming (str): naming convention of files being searched for
            e.g. if looking for all .nii files, do *.nii; if looking for wm seg files, do c2*.nii
        subdir (str): default is None; subdirectory (within subject's folder) to look inside for week folder/file
    """
    parent, level = parent_lookup(file_path, subj_id)

    # folder to search inside: subj_id/subdir/ or subj_id/
    folder_root = parent/subdir if subdir else parent

    # depth of file
    rel_parts = Path(file_path).relative_to(folder_root).parts # get all components of file path
    depth = len(rel_parts)

    # then goes down *n* folders to find the file
    glob_pattern = "/".join(["*"] * (depth - 1) + [file_naming])

    # return file (if found) as PosixPath
    return [p for p in folder_root.glob(glob_pattern) if p.is_file()]

In [60]:
find_week_files(ref_img, subj_id, file_naming = "c2*.nii")

# e.g. c2*.nii to find all files that start with c2 and end in .nii

# using * for file naming to find all files in that subject's folder

# if using subdir: returns only files in that directory (subdirectory to subj_id directory)
# if not using subdir: returns all files in that subject's directory

[PosixPath('/cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W0/c2CU_2538_W0_T1.nii'),
 PosixPath('/cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W4/c2CU_2538_W4_T1.nii')]

For `sufficient_weeks` loop: we should use the path from `parent_lookup` as the path to look for files in.

It needs to iterate through the weeks, so look for week folders or week files.

So perhaps check all levels?

We can track how many levels *up* it went (from reference image) to find the subject-parent folder and go down that many levels to find the file for each week. So we would need a week-loop in there. Perhaps we can literally just have a list of all *potential* weeks and it'll look for the file for each of those weeks - if it doesn't exist, skip.